# 10 — Pull Archigos Leader Dataset

**Source:** Archigos: A Dataset of Political Leaders (v4.1)  
**Provider:** Goemans, Gleditsch & Chiozza (University of Rochester / LSE)  
**Access:** Public download from Goemans' Rochester page (Stata `.dta`)  
**Coverage:** Global, 1875–2015 (v4.1); selected updates to 2018 in some mirrors  
**Frequency:** Leader-spell level (entry → exit per leader per country)  

## What this notebook pulls

Archigos records every national political leader — their entry/exit dates, how they left
office, gender, military background, and age. This feeds **Domain 2 — Governance/Institutions**
predictors in the meta-plan (coup onset, irregular leadership change).

### Key variables

| Variable | Description |
|---|---|
| `obsid` | Observation ID (country-leader spell) |
| `ccode` | Correlates of War country code |
| `idacr` | 3-letter country acronym |
| `leader` | Leader name |
| `startdate` | Date entered office |
| `enddate` | Date left office (or censored) |
| `exit` | How the leader left: Regular, Irregular, Still in office, Natural death, etc. |
| `exitcode` | Detailed exit code |
| `prevtimesinoffice` | Times previously held office |
| `posttenurefate` | Post-tenure fate (exile, imprisonment, death, etc.) |
| `gender` | Leader gender (0=male, 1=female) |
| `yrborn` | Year of birth |
| `military` | Military background (0/1) |

### Derived country-year panel variables
- `is_leader_change` — any leadership change occurred this year
- `irregular_exit_count` — number of irregular exits in country-year
- `regular_exit_count` — number of regular exits in country-year
- `n_leaders` — number of distinct leaders active this year
- `has_military_leader` — any active leader has military background
- `leader_tenure_years` — mean tenure (years) of leaders active this year

## Access note
The Archigos dataset is freely available. If the primary URL fails, place the Stata
file manually at `archigos_files/archigos.dta` and re-run from the **Load data** cell.

```
archigos_files/archigos.dta     ← Stata format (v4.1)
```

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import zipfile
import requests
import pandas as pd
import pyreadstat
from pathlib import Path
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Archigos v4.1 — Stata file bundled in a zip archive
# Primary: Goemans' Rochester page
ARCHIGOS_URL = (
    "http://www.rochester.edu/college/faculty/hgoemans/"
    "Archigos_4.1.zip"
)

# Local fallback
ARCHIGOS_LOCAL_DIR  = Path("archigos_files")
ARCHIGOS_LOCAL_FILE = ARCHIGOS_LOCAL_DIR / "archigos.dta"

print(f"Run date        : {RUN_DATE}")
print(f"Primary URL     : {ARCHIGOS_URL}")
print(f"Local fallback  : {ARCHIGOS_LOCAL_FILE} (exists={ARCHIGOS_LOCAL_FILE.exists()})")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Load Archigos data

Tries to download the zip from Rochester, extracts the `.dta` file, and
reads it via `pyreadstat`. Falls back to a local file if the URL is unavailable.

In [ ]:
def load_archigos_from_url(url: str) -> pd.DataFrame | None:
    print(f"Downloading Archigos from {url} ...")
    try:
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Download failed: {e}")
        return None

    content = resp.content

    # The file may be a zip archive containing a .dta file
    if url.lower().endswith(".zip") or resp.headers.get("content-type", "").startswith("application/zip"):
        try:
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                dta_name = next((n for n in z.namelist() if n.lower().endswith(".dta")), None)
                if dta_name is None:
                    # fallback: try CSV inside zip
                    csv_name = next((n for n in z.namelist() if n.lower().endswith(".csv")), None)
                    if csv_name:
                        with z.open(csv_name) as f:
                            return pd.read_csv(f, low_memory=False)
                    print("  No .dta or .csv file found inside zip")
                    return None
                with z.open(dta_name) as f:
                    raw_bytes = f.read()
            df, meta = pyreadstat.read_dta(io.BytesIO(raw_bytes))
            return df
        except Exception as e:
            print(f"  Failed to extract/read zip: {e}")
            return None
    else:
        # Assume direct .dta download
        try:
            df, meta = pyreadstat.read_dta(io.BytesIO(content))
            return df
        except Exception as e:
            print(f"  Failed to read .dta bytes: {e}")
            return None


def load_archigos_from_local(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    print(f"Loading Archigos from local file: {path}")
    try:
        df, meta = pyreadstat.read_dta(str(path))
        return df
    except Exception as e:
        print(f"  pyreadstat failed ({e}); trying pandas")
        try:
            return pd.read_stata(str(path))
        except Exception as e2:
            print(f"  pandas read_stata also failed: {e2}")
            return None


df_archigos_raw = load_archigos_from_url(ARCHIGOS_URL)

if df_archigos_raw is None:
    df_archigos_raw = load_archigos_from_local(ARCHIGOS_LOCAL_FILE)

if df_archigos_raw is None:
    print(
        "\nWARNING: Archigos data not available.\n"
        "  Download Archigos_4.1.zip from:\n"
        "  http://www.rochester.edu/college/faculty/hgoemans/\n"
        f"  and place the .dta file at: {ARCHIGOS_LOCAL_FILE}\n"
    )
else:
    df_archigos_raw.columns = [c.lower().strip() for c in df_archigos_raw.columns]
    print(f"Archigos raw: {len(df_archigos_raw):,} leader-spells | columns: {list(df_archigos_raw.columns)}")

## Schema and type casting

Archigos uses numeric codes for categorical variables (exit, gender, military).
We preserve the raw numeric codes and add human-readable string columns.

In [ ]:
EXPECTED_COLS = [
    "obsid", "ccode", "idacr", "leader",
    "startdate", "enddate",
    "exit", "exitcode",
    "prevtimesinoffice", "posttenurefate",
    "gender", "yrborn", "military",
]

if df_archigos_raw is not None:
    missing = [c for c in EXPECTED_COLS if c not in df_archigos_raw.columns]
    if missing:
        print(f"WARNING: expected columns not found: {missing}")
    present = [c for c in EXPECTED_COLS if c in df_archigos_raw.columns]
    df_arch = df_archigos_raw[present].copy()

    # Parse date columns — Archigos dates may come as strings (YYYY-MM-DD)
    for dcol in ["startdate", "enddate"]:
        if dcol in df_arch.columns:
            df_arch[dcol] = pd.to_datetime(df_arch[dcol], errors="coerce")

    # Derive start/end years and tenure
    if "startdate" in df_arch.columns:
        df_arch["start_year"] = df_arch["startdate"].dt.year.astype("Int64")
    if "enddate" in df_arch.columns:
        df_arch["end_year"] = df_arch["enddate"].dt.year.astype("Int64")
    if "startdate" in df_arch.columns and "enddate" in df_arch.columns:
        df_arch["tenure_days"] = (df_arch["enddate"] - df_arch["startdate"]).dt.days
        df_arch["tenure_years"] = (df_arch["tenure_days"] / 365.25).round(2)

    # Cast integer columns
    for icol in ["ccode", "gender", "military", "yrborn", "prevtimesinoffice"]:
        if icol in df_arch.columns:
            df_arch[icol] = pd.to_numeric(df_arch[icol], errors="coerce").astype("Int64")

    # Irregular exit flag: Archigos codes 'exit' as 'Irregular' string or numeric 3/4
    if "exit" in df_arch.columns:
        exit_str = df_arch["exit"].astype(str).str.strip().str.lower()
        df_arch["irregular_exit"] = (
            exit_str.str.contains("irregular", na=False)
        ).astype("Int8")

    print(f"Archigos cleaned: {len(df_arch):,} leader-spells")
    print(f"Columns: {list(df_arch.columns)}")
    df_arch.head(3)

## Write leader-spell data to ADLS

In [ ]:
if df_archigos_raw is not None and not df_arch.empty:
    write_parquet(df_arch, f"raw/archigos/{RUN_DATE}/archigos_spells.parquet")

## Derive country-year panel

The panel model works at **country-year** granularity. We expand each leader spell
across the years they were in office and aggregate to country-year summaries.

In [ ]:
if df_archigos_raw is not None and not df_arch.empty and "start_year" in df_arch.columns:
    records = []

    for _, row in df_arch.iterrows():
        sy = row.get("start_year")
        ey = row.get("end_year")
        if pd.isna(sy):
            continue
        if pd.isna(ey):
            ey = sy  # still in office — count only entry year
        sy, ey = int(sy), int(ey)

        for yr in range(sy, ey + 1):
            records.append({
                "ccode":          row.get("ccode"),
                "idacr":          row.get("idacr"),
                "year":           yr,
                "leader":         row.get("leader"),
                "is_entry_year":  int(yr == sy),
                "is_exit_year":   int(yr == ey and not pd.isna(row.get("end_year"))),
                "irregular_exit": row.get("irregular_exit", 0),
                "military":       row.get("military"),
                "gender":         row.get("gender"),
                "tenure_years":   row.get("tenure_years"),
            })

    df_spells_expanded = pd.DataFrame(records)

    # Aggregate to country-year
    df_country_year = (
        df_spells_expanded
        .groupby(["ccode", "idacr", "year"], as_index=False)
        .agg(
            n_leaders           =("leader",         "nunique"),
            is_leader_change    =("is_entry_year",  "max"),
            irregular_exit_count=("irregular_exit", "sum"),
            has_military_leader =("military",        "max"),
            has_female_leader   =("gender",          lambda x: int((x == 1).any())),
            mean_tenure_years   =("tenure_years",   "mean"),
        )
    )

    for col in ["n_leaders", "is_leader_change", "irregular_exit_count",
                "has_military_leader", "has_female_leader"]:
        df_country_year[col] = df_country_year[col].astype("Int64")

    print(f"Country-year panel: {len(df_country_year):,} rows")
    print(f"Years: {df_country_year['year'].min()}–{df_country_year['year'].max()}")
    print(f"Countries (ccode): {df_country_year['ccode'].nunique()}")
    df_country_year.head(3)
else:
    df_country_year = pd.DataFrame()

In [ ]:
if not df_country_year.empty:
    write_parquet(df_country_year, f"raw/archigos/{RUN_DATE}/archigos_country_year.parquet")

## Summary

In [ ]:
print("=" * 55)
print("Archigos pull complete")
print("=" * 55)
if df_archigos_raw is not None:
    print(f"  Leader spells     : {len(df_arch):,}")
if not df_country_year.empty:
    print(f"  Country-years     : {len(df_country_year):,}")
    print(f"  Year range        : {df_country_year['year'].min()}–{df_country_year['year'].max()}")
print()
print("ADLS paths written:")
print(f"  raw/archigos/{RUN_DATE}/archigos_spells.parquet")
print(f"  raw/archigos/{RUN_DATE}/archigos_country_year.parquet")